# Day7 答案：完整 pipeline

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

DATA = Path("../data")
compounds = pd.read_csv(DATA / "compounds.csv")
cell_lines = pd.read_csv(DATA / "cell_lines.csv")
sensitivity = pd.read_csv(DATA / "sensitivity.csv")

df = (
    sensitivity.dropna(subset=["IC50"])
    .drop_duplicates()
    .merge(compounds, on="drug_id", how="inner")
    .merge(cell_lines, on="cell_id", how="inner")
    .dropna(subset=["smiles"])
)

def smiles_to_fp(smiles, n_bits=32):
    vec = np.zeros(n_bits)
    tokens = list("CNOSF=#()[]123cno") + ["Cl", "Br"]
    for i, tok in enumerate(tokens):
        vec[i % n_bits] += str(smiles).count(tok)
    vec[-1] = len(str(smiles))
    return vec

X = np.vstack([smiles_to_fp(s) for s in df["smiles"]])
y = df["IC50"].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmses, r2s = [], []
for tr, te in kf.split(X):
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X[tr], y[tr])
    pred = model.predict(X[te])
    rmses.append(np.sqrt(mean_squared_error(y[te], pred)))
    r2s.append(r2_score(y[te], pred))

print("mean RMSE:", float(np.mean(rmses)))
print("mean R2:", float(np.mean(r2s)))
